# Text Captcha Solver (CRNN + CTC)

Reads 5-character text captchas with a CNN + BiLSTM + CTC model. The letters shift
around and overlap each other, which is exactly the case CTC is good at: it works
out the alignment on its own.

To run everything, open the Runtime menu and pick Run all. Cell 3 stops to ask for
`text-captcha-solver.zip` (the project files plus the images). After that it goes
on by itself, first training, then a short demo.

Switch the runtime to GPU before you start (Runtime, Change runtime type, GPU, T4).


## 1) Check the GPU


In [ ]:
!nvidia-smi -L || echo 'No GPU. Use Runtime > Change runtime type > GPU for a big speedup.'

## 2) Dependencies (torch is preinstalled on Colab)


In [ ]:
!pip -q install pillow numpy >/dev/null 2>&1
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())

## 3) Upload and unpack the project
Pick the project zip when the dialog opens. It has the code in it, plus a `data/`
folder and `labels.csv`.


In [ ]:
import os, zipfile, glob
if not os.path.exists('labels.csv'):
    try:
        from google.colab import files
        up = files.upload()   # select the project zip
        zname = [f for f in up if f.endswith('.zip')][0]
        with zipfile.ZipFile(zname) as z: z.extractall('.')
    except Exception as e:
        print('No interactive upload. Upload and unzip the file manually.', e)
print('labels.csv:', os.path.exists('labels.csv'))
print('images in data/:', len(glob.glob('data/*.jpg')))

## 4) Train the CTC model
400 epochs, which runs 15 to 20 minutes on a GPU. The cosine schedule is tied to
whatever epoch count you pass, so shortening the run also compresses the schedule.
`--val 200` keeps 200 images back for testing.


In [ ]:
!python train_ctc.py --data data --labels labels.csv --val 200 --epochs 400 --batch 64 --lr 1e-3 --out ctc_model.pt

## 5) Demo: solve a few images


In [ ]:
import glob
sample = ' '.join(sorted(glob.glob('data/*.jpg'))[:15])
!python solve_ctc.py {sample} --model ctc_model.pt

## 6) Solve your own image
Run this cell and pick a jpg or png off your machine. The prediction shows up next
to the image.


In [ ]:
from google.colab import files
from PIL import Image
import torch, matplotlib.pyplot as plt
from model_ctc import load_model, preprocess, greedy_decode

m = load_model('ctc_model.pt', 'cuda' if torch.cuda.is_available() else 'cpu')
dev = next(m.parameters()).device

up = files.upload()          # select one or more images
for fn in up:
    img = Image.open(fn)
    x = preprocess(img).unsqueeze(0).to(dev)
    with torch.no_grad():
        logits = m(x)
    text = greedy_decode(logits)[0]
    conf = float(logits.softmax(2).max(2).values.mean())
    plt.figure(figsize=(4,1)); plt.imshow(img, cmap='gray'); plt.axis('off')
    plt.title(f'{text}   (conf {conf:.2f})'); plt.show()
    print(fn, '->', text, f'(conf {conf:.2f})')

## 7) Download the model


In [ ]:
try:
    from google.colab import files; files.download('ctc_model.pt')
except Exception as e:
    print('ctc_model.pt is ready.', e)

---
### Notes on accuracy
Once the model has fit the training set the loss goes to zero, and from there
validation accuracy stops climbing and just wobbles inside a band. Letting it run
well past that point buys you nothing. If you want more out of it, the things to
try are cleaning up the hardest labels, test-time augmentation, or a bigger
backbone.

Full-string accuracy sits around 95%, so a retry or two puts the effective success
rate near 100%.
